# The Bayesian t-test: from *is there an effect?* to *how big is it?*

Notebook 01 asked one question: *if H₀ were true, how surprising is our t?* It
answers with a yes/no stamp — reject, or don't. But it never tells us the thing we
usually care about: **how large is the effect, and how sure are we?**

The Bayesian move needs no new machinery — just a change of reading. Take the very
same t-distribution and re-centre it: instead of parking it on 0 (the null) and
asking about our observed t, park it on **our observed difference** and let it
describe the whole range of effect sizes the data find plausible. (With flat priors
this *is* the posterior for the mean difference — a shifted, scaled t.)

Both panels below are drawn on the **same square scale as notebook 01**, so you can
read one against the other. The lower panel is the distribution over the effect
size **Δ**:

| you see | it means |
|---|---|
| the curve's **peak** | the observed difference **Δ̂** = x̄₂ − x̄₁ — our best estimate |
| the curve's **width** | our uncertainty — wide when data are few or noisy |
| the curve's **height** | it's a real density (area = 1), so concentrating ⇒ taller |
| the dashed line at **0** | the null; if it sits out in the tail, the data doubt it |
| the green dashed **Δ** | the real effect — unknown in life, shown here because we simulate |

Because it is a genuine density, a very certain posterior becomes a tall, thin spike —
and when it would run off the top of the box we simply **clip it to a filled block**:
*more certain than the panel can show.*

And the bridge between the two panels: **hover the lower curve.** Pick any candidate
effect size and the top panel shows the two population means it would imply. The
distribution over Δ is, quite literally, a distribution over *how far apart the two
worlds are.*


In [5]:
# The machinery of the demo. bayes_svg() draws both square panels into one SVG
# on the SAME scale as notebook 01; bayes_script() returns the hover wiring,
# fed the same geometry. bayes_html() pairs them with a fresh unique id.
import itertools
import numpy as np
from scipy import stats

_uid = itertools.count()

BLUE, ORANGE = "#3D6FB0", "#E0883A"
GREEN = "#2E7D5B"
INK, MUTED, RULE, PALE = "#333333", "#888888", "#DDDDDD", "#BFBFBF"
FONT = "font-family:system-ui,-apple-system,sans-serif;"

POOL = 100

# --- the shared square + scale (identical to notebook 01) -------------------
S = 350
HALF = 5.0
YMAX = 1.25
SC = S / (2 * HALF)                # = 35.0
K = S / YMAX                       # = 280
ML, MR = 16, 16
FW = ML + S + MR
X0 = ML + S / 2
BAND = 50                          # top strip of the posterior box reserved for labels
CAP = S - BAND                     # curve ceiling there (truncate above this)

# vertical layout of the two stacked panels
POP_TOP = 22
POPB = POP_TOP + S                 # populations baseline
POST_TOP = POPB + 50               # posterior box top
POSTB = POST_TOP + S               # posterior baseline
CEIL = POST_TOP + BAND             # y of the truncation ceiling
FH = POSTB + 24


def make_pool(seed=None):
    rng = np.random.default_rng(seed)
    return dict(z1=rng.standard_normal(POOL), z2=rng.standard_normal(POOL),
                j1=rng.random(POOL), j2=rng.random(POOL))


def X(x):
    return X0 + x * SC


def Y(base, dens, cap=S):
    return base - min(dens * K, cap)


def filled_curve(xs, dens, base, stroke, fill, op, sw=2, cap=S, cid=None):
    pts = [(X(x), Y(base, d, cap)) for x, d in zip(xs, dens)]
    line = " ".join(f"{px:.1f},{py:.1f}" for px, py in pts)
    poly = f"{pts[0][0]:.1f},{base:.1f} {line} {pts[-1][0]:.1f},{base:.1f}"
    idattr = f"id='{cid}' " if cid else ""
    return (f"<polygon points='{poly}' fill='{fill}' opacity='{op}'/>"
            f"<polyline {idattr}points='{line}' fill='none' stroke='{stroke}' stroke-width='{sw}'/>")


def clampx(px, pad=24):
    return min(max(px, ML + pad), ML + S - pad)


# ----------------------------------------------------------------------------
# the combined figure
# ----------------------------------------------------------------------------

def bayes_svg(delta, var, n, pool, hover=None, uid="bayes"):
    sigma = var ** 0.5
    xa = -delta / 2 + sigma * pool["z1"][:n]
    xb = delta / 2 + sigma * pool["z2"][:n]
    d_obs = float(xb.mean() - xa.mean())
    sp = np.sqrt((xa.var(ddof=1) + xb.var(ddof=1)) / 2)
    se = sp * np.sqrt(2 / n)
    df = 2 * n - 2

    p = []

    # ---- top panel: populations + sample -----------------------------------
    xs = np.linspace(-HALF, HALF, 201)
    for mu, col in ((-delta / 2, BLUE), (delta / 2, ORANGE)):
        pdf = np.exp(-0.5 * ((xs - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
        p.append(filled_curve(xs, pdf, POPB, col, col, 0.10))
    p.append(f"<line x1='{ML}' x2='{ML + S}' y1='{POPB}' y2='{POPB}' stroke='{RULE}' stroke-width='1'/>")
    p.append(f"<text x='{ML}' y='{POP_TOP - 6}' style='{FONT}' font-size='10.5' "
             f"letter-spacing='1px' fill='{MUTED}'>POPULATIONS &amp; SAMPLE</text>")
    for z, j, mu, col, side in ((pool["z1"], pool["j1"], -delta / 2, BLUE, -1),
                                (pool["z2"], pool["j2"], delta / 2, ORANGE, 1)):
        for zi, ji in zip(z[:n], j[:n]):
            x = max(-HALF, min(HALF, mu + sigma * zi))
            y = POPB + side * (9 + ji * 6)
            p.append(f"<circle cx='{X(x):.1f}' cy='{y:.1f}' r='2.6' fill='{col}' opacity='0.75'/>")

    # implied-mean guides (hidden until hover) — two dashed verticals + a label
    op = 1 if hover is not None else 0
    xl, xr = X(-(hover or 0) / 2), X((hover or 0) / 2)
    for cid, xv in ((f"impL_{uid}", xl), (f"impR_{uid}", xr)):
        p.append(f"<line id='{cid}' x1='{xv:.1f}' x2='{xv:.1f}' y1='{POPB}' y2='{POP_TOP + 6}' "
                 f"stroke='{INK}' stroke-width='1.2' stroke-dasharray='4 3' opacity='{op}'/>")
    p.append(f"<text id='impLbl_{uid}' x='{X0:.1f}' y='{POP_TOP + 12}' text-anchor='middle' style='{FONT}' "
             f"font-size='10' font-style='italic' fill='{MUTED}' opacity='{op}'>implied means</text>")

    # ---- bottom panel: posterior density over Δ (true area-1 density) -------
    ds = np.linspace(-HALF, HALF, 321)
    dens = stats.t.pdf((ds - d_obs) / se, df) / se
    p.append(filled_curve(ds, dens, POSTB, INK, INK, 0.06, cap=CAP, cid=f"postCurve_{uid}"))
    p.append(f"<line x1='{ML}' x2='{ML + S}' y1='{POSTB}' y2='{POSTB}' stroke='{RULE}' stroke-width='1'/>")
    p.append(f"<text x='{ML}' y='{POST_TOP - 8}' style='{FONT}' font-size='10.5' "
             f"letter-spacing='1px' fill='{MUTED}'>PLAUSIBLE EFFECT SIZES  Δ</text>")

    # three reference markers — all dashed, full height; labels staggered in the
    # reserved top band (and clamped inside) so they breathe and never collide
    # with a truncated spike.
    def marker(xv, color, lbl, lbl_y):
        x = X(xv)
        seg = (f"<line x1='{x:.1f}' x2='{x:.1f}' y1='{POSTB}' y2='{lbl_y + 4:.0f}' "
               f"stroke='{color}' stroke-width='1.4' stroke-dasharray='5 4'/>")
        return seg + (f"<text x='{clampx(x):.1f}' y='{lbl_y}' text-anchor='middle' style='{FONT}' "
                      f"font-size='12.5' font-weight='600' fill='{color}'>{lbl}</text>")

    p.append(marker(delta, GREEN, "Δ", POST_TOP + 14))
    p.append(marker(d_obs, INK, "Δ̂", POST_TOP + 28))
    p.append(marker(0, MUTED, "0", POST_TOP + 42))

    # hover guide on the posterior (hidden until hover); label kept below the
    # ceiling so it never lands in the marker band.
    hx, hy, hlbl_y = X(hover or 0), POSTB, POSTB
    if hover is not None:
        hy = Y(POSTB, stats.t.pdf((hover - d_obs) / se, df) / se, cap=CAP)
        hlbl_y = min(max(hy - 8, CEIL + 10), POSTB - 6)
    p.append(f"<line id='hoverLine_{uid}' x1='{hx:.1f}' x2='{hx:.1f}' "
             f"y1='{POSTB}' y2='{hy:.1f}' stroke='{INK}' stroke-width='1.2' "
             f"stroke-dasharray='4 3' opacity='{op}'/>")
    p.append(f"<text id='hoverLbl_{uid}' x='{clampx(hx):.1f}' y='{hlbl_y:.1f}' text-anchor='middle' "
             f"style='{FONT}' font-size='11' font-weight='600' fill='{INK}' opacity='{op}'>"
             f"Δ = {(hover or 0):.2f}</text>")

    return (f"<svg id='{uid}' width='{FW}' height='{FH}' viewBox='0 0 {FW} {FH}' "
            f"xmlns='http://www.w3.org/2000/svg' style='cursor:crosshair'>"
            + "".join(p) + "</svg>")


# the hover wiring — fed the same geometry constants Python drew with
def bayes_script(uid="bayes"):
    return f"""<script>
(function(){{
  var svg=document.getElementById('{uid}'); if(!svg) return;
  var hov=document.getElementById('hoverLine_{uid}'), hl=document.getElementById('hoverLbl_{uid}');
  var iL=document.getElementById('impL_{uid}'), iR=document.getElementById('impR_{uid}'),
      iLbl=document.getElementById('impLbl_{uid}'), curve=document.getElementById('postCurve_{uid}');
  var pts=curve.getAttribute('points').trim().split(/\\s+/).map(function(s){{
      var a=s.split(','); return [parseFloat(a[0]), parseFloat(a[1])]; }});
  var X0={X0}, SC={SC}, PB={POSTB}, PT={POST_TOP}, CEIL={CEIL}, XMIN={ML}, XMAX={ML + S}, PAD=24;
  function clampx(x){{ return Math.min(Math.max(x, XMIN+PAD), XMAX-PAD); }}
  function cY(x){{ for(var i=1;i<pts.length;i++){{ if(pts[i][0]>=x){{
      var a=pts[i-1],b=pts[i],t=(x-a[0])/((b[0]-a[0])||1); return a[1]+t*(b[1]-a[1]); }} }} return PB; }}
  function usr(e){{ var p=svg.createSVGPoint(); p.x=e.clientX; p.y=e.clientY;
      return p.matrixTransform(svg.getScreenCTM().inverse()); }}
  function hide(){{ [hov,hl,iL,iR,iLbl].forEach(function(el){{ if(el) el.style.opacity=0; }}); }}
  svg.addEventListener('mousemove', function(e){{
      var u=usr(e);
      if(u.x<XMIN||u.x>XMAX||u.y<PT||u.y>PB+18){{ hide(); return; }}
      var x=u.x, cy=cY(x), d=(x-X0)/SC;
      hov.setAttribute('x1',x); hov.setAttribute('x2',x);
      hov.setAttribute('y1',PB); hov.setAttribute('y2',cy); hov.style.opacity=1;
      var ly=Math.min(Math.max(cy-8, CEIL+10), PB-6);
      hl.textContent='Δ = '+d.toFixed(2); hl.setAttribute('x',clampx(x)); hl.setAttribute('y',ly); hl.style.opacity=1;
      var xl=X0-(d/2)*SC, xr=X0+(d/2)*SC;
      iL.setAttribute('x1',xl); iL.setAttribute('x2',xl); iL.style.opacity=1;
      iR.setAttribute('x1',xr); iR.setAttribute('x2',xr); iR.style.opacity=1;
      if(iLbl) iLbl.style.opacity=1;
  }});
  svg.addEventListener('mouseleave', hide);
}})();
</script>"""


def bayes_html(delta, var, n, pool):
    uid = f"b{next(_uid)}"
    return bayes_svg(delta, var, n, pool, uid=uid) + bayes_script(uid)

In [6]:
import ipywidgets as W
from IPython.display import display, HTML, clear_output

def slider(value, mn, mx, step, desc, cls=W.FloatSlider, **kw):
    return cls(value=value, min=mn, max=mx, step=step, description=desc,
               continuous_update=True, style={"description_width": "170px"},
               layout=W.Layout(width="430px"), **kw)

delta_s = slider(0.8, 0.0, 3.0, 0.05, "true difference · Δ", readout_format=".2f")
var_s   = slider(1.0, 0.25, 4.0, 0.05, "true variance · σ²", readout_format=".2f")
n_s     = slider(12, 3, 100, 1, "sample size · n per group", cls=W.IntSlider)

new_btn = W.Button(description="↻  new sample",
                   layout=W.Layout(width="150px", margin="4px 0 0 174px"))

pool = make_pool(seed=15)          # one fixed draw — dots don't jitter as you drag
out = W.Output()

def render(change=None):
    html = bayes_html(delta_s.value, var_s.value, n_s.value, pool)
    with out:
        clear_output(wait=True)
        display(HTML(html))

def resample(_):
    pool.update(make_pool())       # only the button re-randomizes
    render()

for s in (delta_s, var_s, n_s):
    s.observe(render, "value")
new_btn.on_click(resample)
render()

hint = W.HTML(f"<div style='{FONT}font-size:12px;color:{MUTED};font-style:italic;"
              "margin-bottom:4px'>drag a slider — the truth changes; press the button — a "
              "new sample is drawn; <b>hover the lower panel</b> to read off the implied "
              "population means</div>")

W.VBox([hint, delta_s, var_s, n_s, new_btn, out])


## Try this

1. **Hover slowly across the lower curve.** Watch the two implied means on the top
   panel slide apart and back together. The curve is showing you *which separations
   the data consider plausible* — most weight near Δ̂, little out in the tails.
2. **Is 0 under the curve?** With the defaults, barely — almost all the mass sits to
   the right of the null line, so the data clearly prefer a positive effect. (This is
   the Bayesian echo of "significant.") Now drag **Δ toward 0**: the curve drifts left
   until it straddles 0, and "no effect" becomes the most plausible reading.
3. **Raise n.** The curve concentrates — *taller and narrower* — around the truth:
   more data buys certainty about the size, not merely a smaller p. Push n far enough
   and the spike runs off the top and clips to a filled block: certainty off the
   chart. This is what power looks like from the inside.
4. **Raise σ².** The curve fattens and sinks: noise widens the band of effect sizes
   the data can't tell apart.
5. **Press "new sample" a few times.** The curve jumps — its centre Δ̂ is the *noisy*
   observed effect — but it usually keeps the true Δ within its bulk. Each real study
   hands you exactly one of these curves, and you never get to see the green line.

### The moral

* A p-value returns a **verdict**; this returns an **estimate with honest error bars**.
  Same t-distribution, read the other way round: from *"how surprising under H₀"* to
  *"which effects are plausible given the data."*
* The shape is the message. "Significant" can hide a curve so wide it spans trivial
  and enormous effects alike; "not significant" can hide a curve tight around zero.
  The stamp throws that away — the distribution keeps it.
